In [1]:
import os
import json
from langgraph.graph import StateGraph
from langchain.tools import tool
from typing import TypedDict


/home/sathw/.pyenv/versions/3.11.9/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## as mentioned document lookup 

In [2]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader, PyPDFLoader

documents = []
data_dir = Path("./data")

for file in data_dir.iterdir():

    if file.suffix.lower() == ".txt":
        print(f"Loading TXT: {file}")
        documents.extend(TextLoader(str(file)).load())

    elif file.suffix.lower() == ".pdf":
        print(f"Loading PDF: {file}")
        documents.extend(PyPDFLoader(str(file)).load())

print(f"Total documents loaded: {len(documents)}")


/tmp/ipykernel_2059/638227778.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, PyPDFLoader


Loading PDF: data/Retrieval-Augmented_Generation_RAG.pdf
Loading PDF: data/Artificial-Intelligence-in-Healthcare.pdf
Total documents loaded: 42


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks=splitter.split_documents(documents)
print("Splitter created Successfully")
print(f"length of chunks:{len(chunks)}")

Splitter created Successfully
length of chunks:365


In [4]:
metadata={'source': 'data/Retrieval-Augmented_Generation_RAG.pdf', 'page': 1}
metadata={'source': 'data/Artificial-Intelligence-in-Healthcare.pdf', 'page': 1}

In [5]:
from sentence_transformers import SentenceTransformer

model= SentenceTransformer("all-MiniLM-L6-v2")
texts=[chunk.page_content for chunk in chunks]

embeddings=model.encode(
            texts,
            convert_to_numpy=True
        )
print(embeddings.shape)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 590.71it/s]


(365, 384)


In [6]:
import chromadb
client = chromadb.PersistentClient(path='./chroma_base')

collection= client.get_or_create_collection( name= "documents")

collection.add(
     ids=[f"doc_{i}" for i in range(len(texts))],
    documents=texts,
    embeddings=embeddings.tolist(),
    # ids=[f"doc_{i}" for i in range(len(texts))]
    metadatas=[chunk.metadata for chunk in chunks]
)

print("Documents loaded successfully!")

Documents loaded successfully!


In [7]:
def retrieve_documents(query, top_k=5, threshold=1.0):
    """
    Retrieve relevant documents from ChromaDB.
    If no document is similar enough, return an empty list.
    """

    # Create query embedding
    query_embedding = model.encode(query).tolist()

    # Query ChromaDB (include distances for similarity filtering)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]
    distances = results["distances"][0]

    retrieved = []

    # print("\nSimilarity Distances:")
    # print("-" * 50)

    for doc, meta, distance in zip(docs, metas, distances):

        # print(f"Distance: {distance:.4f} | Source: {meta['source']} | Page: {meta['page']}")

        # Keep only relevant chunks
        if distance <= threshold:
            retrieved.append({
                "source": meta["source"],
                "page": meta["page"],
                "content": doc,
                "distance": distance
            })

    if not retrieved:
        print("\nNo relevant documents found in local knowledge base.")
        return []

    return retrieved

In [8]:
# def retrieve_documents(query, top_k=5):

#     query_embedding = model.encode(query).tolist()

#     results = collection.query(
#         query_embeddings=[query_embedding],
#         n_results=top_k
#     )

#     retrieved = []

#     docs = results["documents"][0]
#     metas = results["metadatas"][0]

#     for doc, meta in zip(docs, metas):

#         retrieved.append({
#             "source": meta["source"],
#             "page": meta["page"],
#             "content": doc
#         })

#     return retrieved

In [9]:
doc=retrieve_documents("what is the use of AI in the healthcare?")
for d in doc:
    print(d)

{'source': 'data/Artificial-Intelligence-in-Healthcare.pdf', 'page': 15, 'content': 'is inaccurate,  incomplete, or biased,  the AI may make poor decisions that could lead to harm.  If the \nproduct relies on personal data of individuals, the company would also be required to comply with data \nprotection and cybersecurity laws to ensure data security and prevent breaches.\nAI systems, especially autonomous ones, are increasingly being integrated into healthcare  decision-making.', 'distance': 0.5651127696037292}
{'source': 'data/Artificial-Intelligence-in-Healthcare.pdf', 'page': 7, 'content': 'such as algorithmic changes, potential bias, and the interpretation of AI-driven decisions. 8 Building on these \nefforts, in 2022, the UK’s Regulatory Horizons Council released a report titled The Regulation of AI as a Medical \nDevice, which emphasizes  improving  communication  and encouraging  patient and public involvement \nthroughout the lifecycle of AI medical devices.\n 6 Available at:

In [10]:
doc=retrieve_documents("who is prem Boinpally?")
for d in doc:
    print(d)


No relevant documents found in local knowledge base.


In [11]:
import os
from dotenv import load_dotenv

load_dotenv("myenv.env")

# AWS Credentials
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION")


In [12]:
from langchain_aws import ChatBedrockConverse

llm = ChatBedrockConverse(
    model="amazon.nova-micro-v1:0",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)

In [13]:
@tool
def document_lookup(query: str):
    """ 
     Search uploaded research documents using the local ChromaDB.
    Returns the most relevant document chunks.
    """
    return retrieve_documents(query)

In [14]:
document_lookup.invoke("What are the applications of AI in healthcare?")

[{'source': 'data/Artificial-Intelligence-in-Healthcare.pdf',
  'page': 1,
  'content': 'Artificial Intelligence in Healthcare — Navigating Regulatory Frontiers in India \n© Nishith Desai Associates 2025 Provided upon request only    3\nApplications of AI in Healthcare and \nLife Sciences\n 1 Accessible at: https://pmc.ncbi.nlm.nih.gov/articles/PMC2464549/ . \n 2 Accessible at: https://www.researchgate.net/publication/355069925_Applications_of_Artificial_Intelligence_AI_in_healthcare_A_review .',
  'distance': 0.552137017250061},
 {'source': 'data/Artificial-Intelligence-in-Healthcare.pdf',
  'page': 3,
  'content': 'Artificial Intelligence in Healthcare — Navigating Regulatory Frontiers in India \n© Nishith Desai Associates 2025 Provided upon request only    5\n Applications of AI in Healthcare and Life  Sciences  \nH. Virtual Health Assistants\nAI-driven  virtual  assistants,  such as chatbots and voice-based systems,  provide patients with medical \ninforma tion, answer questions, a

In [15]:
from ddgs import DDGS

@tool
def web_search(query):
    """ Search the web for recent information"""
    with DDGS() as ddgs:
        result =list(ddgs.text(query, max_results=5))

    # print(result)
    
    if not result:
        return []
    
    return result
    # print(result)

In [16]:
web_search.invoke("What are the applications of AI in healthcare?")

[{'title': 'We Take Our Whiskey Neat: Key Insights from AI in Healthcare',
  'href': 'https://www.linkedin.com/pulse/we-take-our-whiskey-neat-key-insights-from-ai-hans-mulder-a3e0c',
  'body': 'AI may make decisions without sufficient human oversight, which can be dangerous in critical healthcare situations. Misinterpretation of complex data or failure to consider individual patient nuances can lead to improper diagnoses or treatment plans.'},
 {'title': 'Healthcare Goes High-Tech: AI Applications for Improved Diagnosis...',
  'href': 'https://ai.plainenglish.io/healtshcare-goes-high-tech-ai-applications-for-improved-diagnosis-and-treatment-65f7854eb97',
  'body': '1. AI can help with diagnosing diseases and creating personalized treatment plans. Visiting a medical practitioner and getting a diagnosis and remedy designed specifically for your well-being in mere minutes? That is the power of Artificial Intelligence in healthcare.'},
 {'title': 'Exploring the Boundless Potential of AI in

In [17]:
import asyncio
from mcp import ClientSession
from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client


In [18]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(
    command="uvx",
    args=["mcp-server-calculator"]
)


In [43]:
from langchain_core.tools import tool

@tool
async def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""

    async with stdio_client(server_params) as (read, write):

        async with ClientSession(read, write) as session:

            await session.initialize()

            result = await session.call_tool(
                "calculate",
                {
                    "expression": expression
                }
            )

            return str(result)

In [44]:
answer = await calculator.ainvoke(
    {
        "expression": "25*(10+5)"
    }
)

print(answer)

meta=None content=[TextContent(type='text', text='375', annotations=None, meta=None)] structuredContent={'result': '375'} isError=False


In [21]:
from pydantic import BaseModel
from typing import List, Dict, Any

class SummaryInput(BaseModel):
    document_results: List[Dict[str, Any]]
    web_results: List[Dict[str, Any]]

In [22]:
from langchain_core.tools import tool

@tool(args_schema=SummaryInput)
def summarize_info(
    document_results: List[Dict[str, Any]],
    web_results: List[Dict[str, Any]]
):
    """
    Summarize retrieved document evidence and web search results.
    """

    # ---------------------------
    # Format Document Results
    # ---------------------------
    doc_text = ""

    for doc in document_results:
        doc_text += f"""
Source: {doc['source']}
Page: {doc['page']}

Content:
{doc['content']}

-------------------------
"""

    # ---------------------------
    # Format Web Results
    # ---------------------------
    web_text = ""

    for result in web_results:
        web_text += f"""
Title: {result['title']}

Summary:
{result['body']}

URL:
{result['href']}

-------------------------
"""

    # ---------------------------
    # Build Prompt Dynamically
    # ---------------------------

    if document_results and web_results:

        prompt = f"""
You are an expert research assistant.

Relevant information was found in BOTH the local knowledge base
and the web.

Document Evidence:
{doc_text}

Web Evidence:
{web_text}

Instructions:
- Combine information from both sources.
- Remove duplicate information.
- Prefer the document when both sources say the same thing.
- Mention important findings.
- Mention differences if any.
- Produce a concise research summary.

Research Summary:
"""

    elif document_results:

        prompt = f"""
You are an expert research assistant.

Only relevant local documents were found.

Document Evidence:
{doc_text}

Instructions:
- Summarize ONLY the document evidence.
- Mention important findings.
- Produce a concise research summary.

Research Summary:
"""

    elif web_results:

        prompt = f"""
You are an expert research assistant.

No relevant local documents were found.

Use ONLY the following web search results.

Web Evidence:
{web_text}

Instructions:
- Summarize ONLY the web information.
- Mention important findings.
- Produce a concise research summary.

Research Summary:
"""

    else:

        return "No relevant information was found in either the local documents or the web."

    response = llm.invoke(prompt)

    return response.content

In [ ]:
from typing import TypedDict

class ResearchState(TypedDict):
    question: str
    plan: str
    documents: list
    web_results: list
    calculator_result: str
    summary: str
    approved: bool
    final_answer: str

In [79]:
def planner_node(state: ResearchState):

    question = state["question"]

    prompt = f"""
    Create a research plan for:

    {question}

    Steps:
    1. Search local documents
    2. Search the web
    3. Summarize
    4. Generate final answer
    5. calculate the expression
    """

    response = llm.invoke(prompt)

    state["plan"] = response.content

    return state

In [71]:
import re

async def retrieval_node(state):

    query = state["question"]

    print("Original Question:", repr(query))

    expression = "".join(
        re.findall(r"[0-9+\-*/().%]+", query)
    )

    print("Expression Sent:", repr(expression))

    calculator_result = await calculator.ainvoke(
        {"expression": expression}
    )

    print("Calculator Result:", calculator_result)

    return {
        **state,
        "documents": [],
        "web_results": [],
        "calculator_result": calculator_result
    }

In [26]:
# def retrieval_node(state: ResearchState):

#     query = state["question"]

#     documents = document_lookup.invoke(query)

#     web = web_search.invoke(query)

#     state["documents"] = documents

#     state["web_results"] = web

#     return state

In [ ]:
def summarizer_node(state):

    summary = summarize_info.invoke(
        {
            "document_results": state["documents"],
            "web_results": state["web_results"],
            "calculator_result": state["calculator_result"]
        }
    )

    if hasattr(summary, "content"):
        state["summary"] = summary.content
    else:
        state["summary"] = summary

    return state

In [ ]:
def approval_node(state: ResearchState):

    print("\n" + "=" * 70)
    print("RETRIEVED SOURCES")
    print("=" * 70)

    # -------------------------
    # Document Sources
    # -------------------------
    if state["documents"]:

        print("\n Document Sources:")

        grouped_docs = {}

        for doc in state["documents"]:
            source = doc["source"]
            page = doc["page"]

            if source not in grouped_docs:
                grouped_docs[source] = []

            grouped_docs[source].append(page)

        for source, pages in grouped_docs.items():
            pages = sorted(set(pages))
            print(f"• {source}")
            print(f"  Pages Retrieved : {pages}")

    else:
        print("\n No relevant local documents found.")

    # -------------------------
    # Web Sources
    # -------------------------
    if state["web_results"]:

        print("\n Web Sources:")

        for web in state["web_results"]:
            print(f"• {web['title']}")

    else:
        print("\n No web results found.")
        
    #calculator

    if state.get("calculator_result"):

        print("\n Calculator Result:")
        print(state["calculator_result"])

    # -------------------------
    # Generated Summary
    # -------------------------
    print("\n" + "=" * 70)
    print("GENERATED RESEARCH SUMMARY")
    print("=" * 70)

    print(state["summary"])

    print("=" * 70)

    # -------------------------
    # Human Approval
    # -------------------------
    while True:

        ans = input("\nApprove this summary and generate the final answer? (yes/no): ").strip().lower()

        if ans in ["yes", "y"]:
            state["approved"] = True
            break

        elif ans in ["no", "n"]:
            state["approved"] = False
            print("\nSummary not approved. Research process stopped.")
            break

        else:
            print("Please enter 'yes'/'y' or 'no'/'n'.")

    return state

In [ ]:
def final_answer_node(state):

    prompt = f"""
Question:
{state['question']}

Summary:
{state['summary']}

Generate a detailed research report.
"""

    response = llm.invoke(prompt)

    if hasattr(response, "content"):
        state["final_answer"] = response.content
    else:
        state["final_answer"] = str(response)

    return state

In [52]:
from langgraph.graph import END
def approval_router(state: ResearchState):

    if state["approved"]:
        return "final"

    return END

In [ ]:
from langgraph.graph import StateGraph, START, END
builder = StateGraph(ResearchState)


builder.add_node("planner", planner_node)
builder.add_node("retriever", retrieval_node)
builder.add_node("summarizer", summarizer_node)
builder.add_node("approval", approval_node)
builder.add_node("final", final_answer_node)

builder.add_edge(START, "planner")
builder.add_edge("planner", "retriever")
builder.add_edge("retriever", "summarizer")
builder.add_edge("summarizer", "approval")

builder.add_conditional_edges(
    "approval",
    approval_router,
    {
        "final": "final",
        END: END
    }
)

builder.add_edge("final", END)

graph = builder.compile()

In [ ]:
# due to mcp of caluclator as using async because that works better in the ipynb type so we are converting the whole tools to async in retreival node due to it we have to convert invoke to await graph.ainvoke because converting into asynchronous
result=await graph.ainvoke({ 
    "question": "What are the applications of AI in healthcare?"
})
# print(result['final_answer'])
if result.get("approved", False):
    print("\n========== FINAL ANSWER ==========\n")
    print(result["final_answer"])
else:
    print("\n========== PROCESS STOPPED ==========\n")
    print("Summary was not approved by the user.")
    print("Final answer was not generated.")


RETRIEVED SOURCES

📄 Document Sources:
• data/Artificial-Intelligence-in-Healthcare.pdf
  Pages Retrieved : [0, 1, 3, 5]

 Web Sources:
• AI for Healthcare Strategies - AWS AI for Healthcare
• AI in Health Care: Applications, Benefits, and Examples
• The Rise of AI in Healthcare
• 25 Healthcare AI Use Cases with Examples
• 6 Applications of AI in Healthcare: Real-World Examples and ...

GENERATED RESEARCH SUMMARY
## Research Summary: Applications and Regulatory Aspects of Artificial Intelligence in Healthcare

### Introduction
Artificial Intelligence (AI) is revolutionizing the healthcare sector by enhancing diagnostic accuracy, personalizing treatment plans, and improving operational efficiencies. This summary synthesizes information from both a local document from Nishith Desai Associates and web resources to provide a comprehensive overview of AI applications, regulatory considerations, and real-world examples.

### Key Applications of AI in Healthcare

#### Virtual Health Assistan

In [60]:
result=await graph.ainvoke({
    "question": "Who is prem Boinpally?"
})
print(result['final_answer'])
if result.get("approved", False):
    print("\n========== FINAL ANSWER ==========\n")
    print(result["final_answer"])
else:
    print("\n========== PROCESS STOPPED ==========\n")
    print("Summary was not approved by the user.")
    print("Final answer was not generated.")


No relevant documents found in local knowledge base.

RETRIEVED SOURCES

📄 No relevant local documents found.

 Web Sources:
• Prem Boinpally - Oliver Wyman | LinkedIn
• Prem Boinpally - Crunchbase Person Profile
• Prem Boinpally - Investor Profile | Connect with top investors
• Cbi: Abhishek Boinpally: TRS founding member's son... - Times of India
• Businessman Abhishek Boinpally Held in Delhi Liquor Scam... - News18

GENERATED RESEARCH SUMMARY
### Research Summary on Prem Boinpally

#### Overview
Prem Boinpally is an individual noted for his professional and investment activities, as well as his family connections within political and business circles.

#### Professional Background
- **LinkedIn Profile:** Prem Boinpally's LinkedIn profile indicates his presence in a professional community with over 1 billion members. It's suggested that Prem might be approached for networking or professional introductions.
- **Investment Activity:** According to Crunchbase, Prem Boinpally made an in

In [61]:
result =await graph.ainvoke({
    "question": "Advantages of RAG?"
})

if result.get("approved", False):
    print("\n========== FINAL ANSWER ==========\n")
    print(result["final_answer"])
else:
    print("\n========== PROCESS STOPPED ==========\n")
    print("Summary was not approved by the user.")
    print("Final answer was not generated.")


RETRIEVED SOURCES

📄 Document Sources:
• data/Retrieval-Augmented_Generation_RAG.pdf
  Pages Retrieved : [1, 4, 5, 6, 7]

 Web Sources:
• Advantages of franchising
• 5 key features and benefits of retrieval augmented generation (RAG)
• What is Retrieval-Augmented Generation (RAG) - GeeksforGeeks
• The Limitations and Advantages of Retrieval Augmented Generation (RAG)
• 7 Key Benefits of RAG in 2026 - stackai.com

GENERATED RESEARCH SUMMARY
**Research Summary on Retrieval-Augmented Generation (RAG)**

**Introduction:**
Retrieval-Augmented Generation (RAG) is a sophisticated framework designed to enhance the reliability and accuracy of AI-generated responses by integrating external information sources into the generation process. RAG is particularly effective in scenarios requiring context-specific and up-to-date information.

**Key Features and Benefits:**

1. **Enhanced Factual Accuracy:**
   - RAG enhances the factual accuracy of AI responses by incorporating external and contemporar

In [83]:
result = await calculator.ainvoke(
    {
        "expression": "(25 + 30)/5 -2"
    }
)
print(result)


meta=None content=[TextContent(type='text', text='9.0', annotations=None, meta=None)] structuredContent={'result': '9.0'} isError=False
